## Setup

In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

## Load Silver

In [17]:
SILVER_PATH = Path("../data/silver/springboks_matches.parquet")
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(SILVER_PATH).sort_values("date").reset_index(drop=True)

## Lagged features

In [18]:
df["rolling_form_3"] = df["win"].shift(1).rolling(3, min_periods=1).mean()
df["rolling_form_5"] = df["win"].shift(1).rolling(5, min_periods=1).mean()
df["rolling_margin_3"] = df["score_margin"].shift(1).rolling(3, min_periods=1).mean()

df["days_since_prev"] = df["date"].diff().dt.days

## Head-to-head feature

In [19]:
prev_wins = {}
prev_games = {}
h2h_values = []

for _, row in df.iterrows():
    opponent = row["opponent"]

    if prev_games.get(opponent, 0) > 0:
        h2h_values.append(prev_wins.get(opponent, 0) / prev_games[opponent])
    else:
        h2h_values.append(np.nan)

    prev_games[opponent] = prev_games.get(opponent, 0) + 1
    prev_wins[opponent] = prev_wins.get(opponent, 0) + int(row["win"] == 1)

df["h2h_winrate"] = h2h_values

## Save Gold and split

In [20]:
# handle missing values from lagged features explicitly
df["rolling_form_3"] = df["rolling_form_3"].fillna(0.5)
df["rolling_form_5"] = df["rolling_form_5"].fillna(0.5)
df["rolling_margin_3"] = df["rolling_margin_3"].fillna(0)
df["h2h_winrate"] = df["h2h_winrate"].fillna(0.5)
df["days_since_prev"] = df["days_since_prev"].fillna(df["days_since_prev"].median())

df_gold = df.reset_index(drop=True)

gold_path = GOLD_DIR / "gold_results.parquet"
df_gold.to_parquet(gold_path, index=False)

TRAIN_END = pd.Timestamp("2016-12-31")

train = df_gold[df_gold["date"] <= TRAIN_END].copy()
test = df_gold[df_gold["date"] > TRAIN_END].copy()

model_columns = [
    "date",
    "home",
    "rolling_form_3",
    "rolling_form_5",
    "rolling_margin_3",
    "h2h_winrate",
    "days_since_prev",
    "opponent",
    "tournament",
    "win"
]

train[model_columns].to_parquet(GOLD_DIR / "train.parquet", index=False)
test[model_columns].to_parquet(GOLD_DIR / "test.parquet", index=False)

print(f"[GOLD] Saved: {gold_path}")
print(f"[GOLD] Rows: {len(df_gold)}")
print(f"[GOLD] Split: train={len(train)}, test={len(test)}")
print(f"[GOLD] Train period: {train['date'].min().date()}–{train['date'].max().date()}")
print(f"[GOLD] Test period: {test['date'].min().date()}–{test['date'].max().date()}")

df_gold.head()

[GOLD] Saved: ../data/gold/gold_results.parquet
[GOLD] Rows: 312
[GOLD] Split: train=255, test=57
[GOLD] Train period: 1992-08-15–2016-11-26
[GOLD] Test period: 2017-06-10–2022-11-26


,date,year,team,opponent,springboks_score,opponent_score,score_margin,win,draw,home,neutral,tournament,stadium,city,country,rolling_form_3,rolling_form_5,rolling_margin_3,days_since_prev,h2h_winrate
0,1992-08-15,1992,South Africa,New Zealand,24,27,-3,0,0,1,False,1992 New Zealand tour,Ellis Park,Johannesburg,South Africa,0.500000,0.500000,0.000000,7.0,0.5
1,1992-08-29,1992,South Africa,Australia,3,26,-23,0,0,1,False,1992 Australia tour of South Africa,Newlands Stadium,Cape Town,South Africa,0.000000,0.000000,-3.000000,14.0,0.5
2,1992-10-17,1992,South Africa,France,20,15,5,1,0,0,False,1992 South Africa rugby union tour of France a...,Stade de Gerland,Lyon,France,0.000000,0.000000,-13.000000,49.0,0.5
3,1992-10-24,1992,South Africa,France,16,29,-13,0,0,0,False,1992 South Africa rugby union tour of France a...,Parc des Princes,Paris,France,0.333333,0.333333,-7.000000,7.0,1.0
4,1992-11-14,1992,South Africa,England,16,33,-17,0,0,0,False,1992 South Africa rugby union tour of France a...,Twickenham,London,England,0.333333,0.250000,-10.333333,21.0,0.5
